In [ ]:
from pathlib import Path
import shutil

# Archivo original en Google Drive
origen = Path(
    "/content/drive/MyDrive/Colab Notebooks/Simulacion_Rutas_Inundacion.ipynb"
)

# Archivo dentro del repositorio Git
destino = Path(
    "/content/Practica02_Inundacion/Simulacion_Rutas_Inundacion.ipynb"
)

def sincronizar_notebook():
    if not origen.is_file():
        raise FileNotFoundError(
            f"No se encontró el notebook: {origen}"
        )

    if not (destino.parent / ".git").is_dir():
        raise RuntimeError("El repositorio no está clonado")

    shutil.copy2(origen, destino)
    print("Notebook copiado al repositorio correctamente")

In [ ]:
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")

if token:
    print("Token cargado correctamente")
else:
    print("No se encontró el token")

In [ ]:
from google.colab import userdata
from pathlib import Path
import os
import subprocess

# Obtener token guardado en Secrets
token = userdata.get("GITHUB_TOKEN")

if not token:
    raise ValueError("No se encontró GITHUB_TOKEN")

# Configurar un helper de Git que consulta las credenciales
# desde la memoria del proceso, sin escribir el token en disco
import base64

encoded = base64.b64encode(
    f"x-access-token:{token}".encode()
).decode()

# Configuración del repositorio para operaciones HTTPS con GitHub
os.environ["GIT_CONFIG_COUNT"] = "1"
os.environ["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
os.environ["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {encoded}"

print("Autenticación de Git configurada para esta sesión.")

# **Crear la matriz de inundación:**

In [ ]:
# Dimensiones del terreno
FILAS = 10
COLUMNAS = 10

# Crear una matriz de 10x10 con zonas seguras
terreno = [
    [0 for columna in range(COLUMNAS)]
    for fila in range(FILAS)
]

# Agregar una zona de inundación alta
for columna in range(1, 9):
    terreno[4][columna] = 2

# Agregar una pequeña región bloqueada
for columna in range(3, 7):
    terreno[2][columna] = 3

# Punto inicial y refugio
inicio = (4, 0)
refugio = (4, 9)

# Consultar el terreno como matriz
for fila in terreno:
    print(fila)

## Definir el costo de cada nivel

In [ ]:
# Costos asociados al nivel de inundación
COSTOS = {
    0: 1,
    1: 4,
    2: 9
}

# Nivel que impide el desplazamiento
BLOQUEADO = 3

In [ ]:
sincronizar_notebook()

In [ ]:
!git status

In [ ]:
!git add .

In [ ]:
!git commit -m "Agregar notebook, estructura de evidencias y creacion de matriz y niveles de inundacion"

In [ ]:
!git push origin main

## Función para obtener vecinos válidos

In [ ]:
def obtener_vecinos(matriz, posicion):
    fila, columna = posicion

    movimientos = [
        (-1, 0),  # Arriba
        (1, 0),   # Abajo
        (0, -1),  # Izquierda
        (0, 1)    # Derecha
    ]

    vecinos = []

    for df, dc in movimientos:
        nueva_fila = fila + df
        nueva_columna = columna + dc

        # Verificar límites del terreno
        if (
            0 <= nueva_fila < len(matriz)
            and 0 <= nueva_columna < len(matriz[0])
        ):

            # Comprobar que la zona sea transitable
            if matriz[nueva_fila][nueva_columna] != BLOQUEADO:
                vecinos.append((nueva_fila, nueva_columna))

    return vecinos

In [ ]:
print(obtener_vecinos(terreno, inicio))

## Función para reconstruir el camino

Los algoritmos registrarán desde qué posición llegó el agente a otra posición. Esta función recuperará el recorrido completo.

In [ ]:
def reconstruir_camino(anteriores, inicio, refugio):

    if inicio == refugio:
        return [inicio]

    if refugio not in anteriores:
        return None

    camino = []
    actual = refugio

    while actual != inicio:
        camino.append(actual)
        actual = anteriores[actual]

    camino.append(inicio)

    camino.reverse()

    return camino

## Función para verificar inicio y refugio

Esta función comprueba que los datos iniciales sean válidos. Es importante porque el algoritmo no debería intentar buscar rutas sobre un terreno mal definido o con el refugio situado en una región intransitable.

In [ ]:
def validar_puntos(matriz, inicio, refugio):

    if not matriz or not matriz[0]:
        raise ValueError("La matriz está vacía")

    columnas = len(matriz[0])

    if any(len(fila) != columnas for fila in matriz):
        raise ValueError("La matriz debe ser rectangular")

    for fila in matriz:
        for nivel in fila:
            if nivel not in COSTOS and nivel != BLOQUEADO:
                raise ValueError("Existe un nivel de inundación inválido")

    for nombre, posicion in [
        ("inicio", inicio),
        ("refugio", refugio)
    ]:
        f, c = posicion

        if not (0 <= f < len(matriz) and 0 <= c < columnas):
            raise ValueError(f"El punto {nombre} está fuera del terreno")

        if matriz[f][c] == BLOQUEADO:
            raise ValueError(f"El punto {nombre} está bloqueado")

## Algoritmo **BFS**

Como todos los movimientos cuentan como un paso, BFS garantiza encontrar un recorrido con la menor cantidad de movimientos cuando existe una ruta. Sin embargo, no garantiza el menor costo de riesgo.

In [ ]:
from collections import deque

def bfs(matriz, inicio, refugio):

    validar_puntos(matriz, inicio, refugio)

    # Cola de posiciones pendientes
    cola = deque([inicio])

    # Posiciones descubiertas
    visitados = {inicio}

    # Registro de posiciones anteriores
    anteriores = {}

    # Contador de nodos explorados
    explorados = 0

    while cola:

        actual = cola.popleft()
        explorados += 1

        # Comprobar si llegamos al refugio
        if actual == refugio:
            camino = reconstruir_camino(
                anteriores, inicio, refugio
            )
            return camino, explorados

        # Explorar los vecinos
        for vecino in obtener_vecinos(matriz, actual):

            if vecino not in visitados:

                visitados.add(vecino)

                anteriores[vecino] = actual

                cola.append(vecino)

    # No existe una ruta
    return None, explorados

## Algoritmo Dijkstra

Dijkstra trabaja de manera diferente. En lugar de priorizar las posiciones con menos movimientos, utiliza el costo acumulado desde el inicio.

In [ ]:
import heapq

def dijkstra(matriz, inicio, refugio):

    validar_puntos(matriz, inicio, refugio)

    # Cola de prioridad: (costo, posicion)
    cola = [(0, inicio)]

    # Menor costo conocido para cada posición
    distancias = {inicio: 0}

    # Registro del camino
    anteriores = {}

    # Nodos cuyo costo mínimo quedó confirmado
    visitados = set()

    explorados = 0

    while cola:

        costo_actual, actual = heapq.heappop(cola)

        # Evitar procesar posiciones repetidas
        if actual in visitados:
            continue

        visitados.add(actual)
        explorados += 1

        # Comprobar si llegamos al refugio
        if actual == refugio:
            camino = reconstruir_camino(
                anteriores, inicio, refugio
            )
            return camino, explorados

        for vecino in obtener_vecinos(matriz, actual):

            fila, columna = vecino

            # Consultar nivel de inundación
            nivel = matriz[fila][columna]

            # Calcular nuevo costo acumulado
            nuevo_costo = costo_actual + COSTOS[nivel]

            # Revisar si encontramos un camino más barato
            if nuevo_costo < distancias.get(vecino, float("inf")):

                distancias[vecino] = nuevo_costo

                anteriores[vecino] = actual

                heapq.heappush(
                    cola,
                    (nuevo_costo, vecino)
                )

    return None, explorados

## Algoritmo A*

A* utiliza la misma lógica de costo acumulado de Dijkstra, pero agrega una estimación de lo que falta para llegar al refugio.

Utilizaremos la distancia Manhattan como heurística, porque solamente permitimos movimientos horizontales y verticales.

In [ ]:
def heuristica(posicion, refugio):

    fila1, columna1 = posicion
    fila2, columna2 = refugio

    return (
        abs(fila1 - fila2)
        + abs(columna1 - columna2)
    )

Ahora implementamos la búsqueda:

In [ ]:
def a_estrella(matriz, inicio, refugio):

    validar_puntos(matriz, inicio, refugio)

    # Cola: (f, g, posicion)
    cola = [(heuristica(inicio, refugio), 0, inicio)]

    # Costo acumulado desde el inicio
    costos_g = {inicio: 0}

    # Registro de posiciones anteriores
    anteriores = {}

    # Posiciones ya expandidas
    visitados = set()

    explorados = 0

    while cola:

        f_actual, g_actual, actual = heapq.heappop(cola)

        # Ignorar entradas antiguas o ya procesadas
        if actual in visitados:
            continue

        visitados.add(actual)
        explorados += 1

        if actual == refugio:
            camino = reconstruir_camino(
                anteriores, inicio, refugio
            )
            return camino, explorados

        for vecino in obtener_vecinos(matriz, actual):

            fila, columna = vecino

            nivel = matriz[fila][columna]

            # Costo real desde el inicio
            nuevo_g = g_actual + COSTOS[nivel]

            if nuevo_g < costos_g.get(vecino, float("inf")):

                costos_g[vecino] = nuevo_g

                anteriores[vecino] = actual

                # Costo estimado hasta el refugio
                h = heuristica(vecino, refugio)

                # Función de evaluación
                nuevo_f = nuevo_g + h

                heapq.heappush(
                    cola,
                    (nuevo_f, nuevo_g, vecino)
                )

    return None, explorados

## Comprobar una ruta

Esta función comprueba cuatro condiciones: que la ruta empiece y termine donde corresponde, que ninguna posición esté fuera de la matriz, que no atraviese regiones bloqueadas y que no realice movimientos diagonales ni saltos entre casillas.

In [ ]:
def validar_ruta(matriz, camino, inicio, refugio):

    if camino is None:
        return False

    if not camino:
        return False

    if camino[0] != inicio or camino[-1] != refugio:
        return False

    for posicion in camino:

        fila, columna = posicion

        # Verificar los límites
        if not (
            0 <= fila < len(matriz)
            and 0 <= columna < len(matriz[0])
        ):
            return False

        # Verificar bloqueos
        if matriz[fila][columna] == BLOQUEADO:
            return False

    for i in range(len(camino) - 1):

        actual = camino[i]
        siguiente = camino[i + 1]

        # Verificar que se mueva una sola casilla
        distancia = (
            abs(actual[0] - siguiente[0])
            + abs(actual[1] - siguiente[1])
        )

        if distancia != 1:
            return False

    return True

## Calcular el costo real de una ruta

Esta función será utilizada por los tres algoritmos, incluido BFS.

Eso es importante: aunque BFS no considere el riesgo al elegir una ruta, sí podremos medir posteriormente cuánto riesgo acumulado tiene la ruta que encontró.

In [ ]:
def calcular_costo(matriz, camino):

    if camino is None:
        return None

    costo_total = 0

    # No cobramos el costo del punto inicial
    for fila, columna in camino[1:]:

        nivel = matriz[fila][columna]

        costo_total += COSTOS[nivel]

    return costo_total

## Ejecutar y comparar algoritmos

In [ ]:
def comparar_algoritmos(matriz, inicio, refugio):

    algoritmos = {
        "BFS": bfs,
        "Dijkstra": dijkstra,
        "A*": a_estrella
    }

    resultados = {}

    for nombre, algoritmo in algoritmos.items():

        camino, explorados = algoritmo(
            matriz, inicio, refugio
        )

        # Registrar los resultados
        resultados[nombre] = {
            "camino": camino,
            "movimientos": (
                len(camino) - 1 if camino is not None else None
            ),
            "costo": calcular_costo(matriz, camino),
            "explorados": explorados,
            "valido": validar_ruta(
                matriz, camino, inicio, refugio
            )
        }

    # Mostrar la comparación
    print(
        f"{'Algoritmo':<12}"
        f"{'Movimientos':<15}"
        f"{'Costo':<10}"
        f"{'Explorados':<12}"
        f"{'Válido'}"
    )

    print("-" * 60)

    for nombre, datos in resultados.items():

        movimientos = datos["movimientos"]
        costo = datos["costo"]

        print(
            f"{nombre:<12}"
            f"{str(movimientos):<15}"
            f"{str(costo):<10}"
            f"{datos['explorados']:<12}"
            f"{datos['valido']}"
        )

    return resultados

## Primera ejecución

In [ ]:
resultados_base = comparar_algoritmos(
    terreno, inicio, refugio
)

for nombre, datos in resultados_base.items():

    print(f"\nAlgoritmo: {nombre}")
    print(f"Camino: {datos['camino']}")

Escenarios de prueba

In [ ]:
def copiar_terreno(matriz):
    return [fila.copy() for fila in matriz]

Prueba 1. Existe una ruta disponible

In [ ]:
escenario_1 = copiar_terreno(terreno)

resultados_1 = comparar_algoritmos(
    escenario_1, inicio, refugio
)

for nombre, datos in resultados_1.items():

    assert datos["camino"] is not None

    assert datos["valido"]

# BFS debe encontrar la ruta con menos movimientos
assert resultados_1["BFS"]["movimientos"] <= \
       resultados_1["Dijkstra"]["movimientos"]

# Dijkstra y A* deben encontrar el costo mínimo
assert resultados_1["Dijkstra"]["costo"] == \
       resultados_1["A*"]["costo"]

# BFS puede ser más corto, pero más costoso
assert resultados_1["BFS"]["costo"] > \
       resultados_1["Dijkstra"]["costo"]

print("Prueba 1 superada")

Prueba 2. Se bloquea una región de la ruta

In [ ]:
escenario_2 = copiar_terreno(terreno)

ruta_original = resultados_1["Dijkstra"]["camino"]

# Elegir una posición intermedia de la ruta
posicion_bloqueada = ruta_original[1]

fila, columna = posicion_bloqueada

# Convertirla en región bloqueada
escenario_2[fila][columna] = BLOQUEADO

print("Posición bloqueada:", posicion_bloqueada)

resultados_2 = comparar_algoritmos(
    escenario_2, inicio, refugio
)

for nombre, datos in resultados_2.items():

    assert datos["camino"] is not None
    assert datos["valido"]

    # Ninguna ruta puede atravesar la nueva región bloqueada
    assert posicion_bloqueada not in datos["camino"]

print("Prueba 2 superada")

Prueba 3. El refugio se vuelve inaccesible

In [ ]:
escenario_3 = copiar_terreno(terreno)

# Bloquear los tres accesos posibles al refugio
escenario_3[3][9] = BLOQUEADO
escenario_3[5][9] = BLOQUEADO
escenario_3[4][8] = BLOQUEADO

resultados_3 = comparar_algoritmos(
    escenario_3, inicio, refugio
)

for nombre, datos in resultados_3.items():

    assert datos["camino"] is None
    assert datos["costo"] is None
    assert datos["movimientos"] is None

print("Prueba 3 superada: refugio inaccesible")

Prueba 4. Se modifica el riesgo de una región

In [ ]:
escenario_4 = copiar_terreno(terreno)

# Tomar la ruta óptima anterior
ruta_anterior = resultados_1["Dijkstra"]["camino"]

# Elevar el riesgo de sus casillas intermedias
for fila, columna in ruta_anterior[1:-1]:

    if escenario_4[fila][columna] != BLOQUEADO:
        escenario_4[fila][columna] = 2

resultados_4 = comparar_algoritmos(
    escenario_4, inicio, refugio
)

ruta_nueva = resultados_4["Dijkstra"]["camino"]

assert ruta_nueva is not None
assert resultados_4["Dijkstra"]["valido"]

assert resultados_4["Dijkstra"]["costo"] == \
       resultados_4["A*"]["costo"]

# Confirmar que Dijkstra ya no eligió la ruta anterior
assert ruta_nueva != ruta_anterior

print("Ruta anterior:", ruta_anterior)
print("Ruta nueva:", ruta_nueva)

print("Prueba 4 superada")

Prueba 5. El camino más corto deja de ser conveniente

In [ ]:
escenario_5 = copiar_terreno(terreno)

resultados_5 = comparar_algoritmos(
    escenario_5, inicio, refugio
)

bfs_resultado = resultados_5["BFS"]
dijkstra_resultado = resultados_5["Dijkstra"]

# BFS encuentra menos movimientos
assert (
    bfs_resultado["movimientos"]
    < dijkstra_resultado["movimientos"]
)

# Pero Dijkstra encuentra menor costo
assert (
    dijkstra_resultado["costo"]
    < bfs_resultado["costo"]
)

print("\nRuta BFS:")
print(bfs_resultado["camino"])

print("\nRuta Dijkstra:")
print(dijkstra_resultado["camino"])

print("\nPrueba 5 superada")

Prueba 6. Caso especial: inicio igual al refugio

In [ ]:
escenario_6 = copiar_terreno(terreno)

inicio_6 = (0, 0)
refugio_6 = (0, 0)

resultados_6 = comparar_algoritmos(
    escenario_6, inicio_6, refugio_6
)

for nombre, datos in resultados_6.items():

    assert datos["camino"] == [(0, 0)]

    assert datos["movimientos"] == 0

    assert datos["costo"] == 0

    assert datos["explorados"] == 1

    assert datos["valido"]

print("Prueba 6 superada: inicio igual al refugio")

7. Resumen automatizado de las pruebas

In [ ]:
escenarios = {
    "Ruta disponible": resultados_1,
    "Zona bloqueada": resultados_2,
    "Refugio inaccesible": resultados_3,
    "Cambio de riesgo": resultados_4,
    "Ruta corta vs segura": resultados_5,
    "Inicio igual al refugio": resultados_6
}

print(
    f"{'Escenario':<26}"
    f"{'Algoritmo':<12}"
    f"{'Mov.':<8}"
    f"{'Costo':<8}"
    f"{'Explorados'}"
)

print("-" * 70)

for escenario, resultados in escenarios.items():

    for algoritmo, datos in resultados.items():

        print(
            f"{escenario:<26}"
            f"{algoritmo:<12}"
            f"{str(datos['movimientos']):<8}"
            f"{str(datos['costo']):<8}"
            f"{datos['explorados']}"
        )